# Week 11 - Deep Segmentation: U-Net, YOLO-seg, Mask R-CNN and SAM

**MCTE 4323 / MCTA 4364 Machine Vision**

### Learning objectives
By the end of this lab you will be able to:
- Distinguish **semantic**, **instance** and **panoptic** segmentation.
- Explain the **encoder-decoder + skip connection** design of **U-Net**.
- Run and compare modern segmentation systems: **DeepLabV3**, **YOLO-seg**, **Mask R-CNN** and **SAM**.
- Evaluate segmentation with **IoU / mIoU**.

### Three flavours of segmentation
- **Semantic**: every pixel labelled by class (all cars are one blob).
- **Instance**: each object instance separated (car 1 vs car 2).
- **Panoptic**: semantic + instance together.

## 1. Setup
> Downloading several pretrained models. GPU recommended.

In [ ]:
import os

# Works in Colab (clones the repo) and locally or in CI (runs inside the repo)
if not os.path.exists("resources/scripts/cvhelpers.py"):
    if not os.path.isdir("MCTA-4364-Machine-Vision"):
        !git clone https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git
    %cd MCTA-4364-Machine-Vision
!pip -q install torch torchvision ultralytics matplotlib opencv-python ipywidgets

In [ ]:
import sys
sys.path.append("resources/scripts")
import cv2, numpy as np, torch
from PIL import Image
from cvhelpers import show, concept_map
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 2. Segmentation families (concept map)

In [ ]:
concept_map([
    "Semantic segmentation: class per pixel (FCN, U-Net, DeepLab)",
    "Instance segmentation: object + mask per instance (Mask R-CNN, YOLO-seg)",
    "Panoptic segmentation: stuff + things together",
    "Foundation models: promptable segmentation with SAM / SAM2"
], title="From semantic to promptable segmentation")

## 3. Guided example - U-Net architecture
U-Net downsamples (encoder) to capture context, then upsamples (decoder) to recover resolution. **Skip connections** copy high-resolution detail from encoder to decoder - which is why it segments fine structures so well.

In [ ]:
import matplotlib.pyplot as plt

def draw_unet():
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.axis("off")
    enc = [("64", 0.15), ("128", 0.32), ("256", 0.49), ("512", 0.66)]
    for label, x in enc:
        ax.add_patch(plt.Rectangle((x, 0.25), 0.09, 0.5, fc="#e8f0fe", ec="#1a73e8", lw=1.5))
        ax.text(x + 0.045, 0.5, label, ha="center", va="center", fontsize=9)
    ax.add_patch(plt.Rectangle((0.66, 0.05), 0.09, 0.15, fc="#fce8e6", ec="#d93025", lw=1.5))
    ax.text(0.705, 0.125, "bottleneck", ha="center", va="center", fontsize=8)
    dec = [("512", 0.83), ("256", 0.70), ("128", 0.57), ("64", 0.44)]
    for label, x in dec:
        ax.add_patch(plt.Rectangle((x, 0.25), 0.09, 0.5, fc="#e6f4ea", ec="#188038", lw=1.5))
        ax.text(x + 0.045, 0.5, label, ha="center", va="center", fontsize=9)
    for x in [0.24, 0.41, 0.58, 0.75]:
        ax.annotate("", xy=(x + 0.08, 0.85), xytext=(x - 0.08, 0.85),
                    arrowprops=dict(arrowstyle="-|>", color="#f9ab00", lw=2))
    ax.text(0.5, 0.93, "skip connections (copy & concatenate)", ha="center", color="#b06000", fontsize=10)
    ax.text(0.15, 0.1, "Encoder (context)", ha="center", color="#1a73e8")
    ax.text(0.85, 0.1, "Decoder (resolution)", ha="center", color="#188038")
    ax.set_title("U-Net: encoder-decoder with skip connections", fontweight="bold")
    plt.tight_layout(); plt.show()

draw_unet()

In [ ]:
import torch.nn as nn

def double_conv(ci, co):
    return nn.Sequential(nn.Conv2d(ci, co, 3, padding=1), nn.ReLU(inplace=True),
                         nn.Conv2d(co, co, 3, padding=1), nn.ReLU(inplace=True))

class MiniUNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=2):
        super().__init__()
        self.e1, self.e2 = double_conv(in_ch, 16), double_conv(16, 32)
        self.e3, self.b = double_conv(32, 64), double_conv(64, 128)
        self.u3, self.d3 = nn.ConvTranspose2d(128, 64, 2, 2), double_conv(128, 64)
        self.u2, self.d2 = nn.ConvTranspose2d(64, 32, 2, 2), double_conv(64, 32)
        self.u1, self.d1 = nn.ConvTranspose2d(32, 16, 2, 2), double_conv(32, 16)
        self.out = nn.Conv2d(16, out_ch, 1)
        self.pool = nn.MaxPool2d(2)
    def forward(self, x):
        c1 = self.e1(x); p1 = self.pool(c1)
        c2 = self.e2(p1); p2 = self.pool(c2)
        c3 = self.e3(p2); p3 = self.pool(c3)
        bn = self.b(p3)
        d3 = self.d3(torch.cat([self.u3(bn), c3], dim=1))
        d2 = self.d2(torch.cat([self.u2(d3), c2], dim=1))
        d1 = self.d1(torch.cat([self.u1(d2), c1], dim=1))
        return self.out(d1)

net = MiniUNet()
x = torch.randn(1, 3, 256, 256)
y = net(x)
print("Input :", tuple(x.shape))
print("Output:", tuple(y.shape), "(same spatial size, out_ch classes per pixel)")
print("Parameters:", sum(p.numel() for p in net.parameters()))

## 4. Guided example - semantic segmentation with DeepLabV3
Torchvision ships DeepLabV3 pretrained on **VOC** (21 classes). We colour the class map and overlay it.

In [ ]:
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights
w = DeepLabV3_ResNet50_Weights.DEFAULT
seg_model = deeplabv3_resnet50(weights=w).eval()
voc = w.meta["categories"]

img_bgr = cv2.imread("resources/images/test_image.jpeg")
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
inp = w.transforms()(Image.fromarray(img_rgb)).unsqueeze(0)
with torch.no_grad():
    logits = seg_model(inp)["out"]
pred = logits.argmax(1)[0].cpu().numpy()
print("Classes present:", sorted(set(pred.ravel().tolist())))

rng = np.random.RandomState(42)
palette = rng.randint(0, 255, (len(voc), 3), dtype=np.uint8)
colour = palette[pred]
# Resize the prediction back to the original image size (the model resizes internally)
colour = cv2.resize(colour, (img_rgb.shape[1], img_rgb.shape[0]), interpolation=cv2.INTER_NEAREST)
overlay = cv2.addWeighted(img_rgb, 0.5, colour, 0.5, 0)
show(colour, overlay, titles=["DeepLabV3 class map", "Overlay"], figsize=(12, 6))

## 5. Guided example - instance segmentation with YOLO-seg
YOLO-seg extends YOLO with a mask head, giving one mask **per detected instance** - fast enough for real-time.

In [ ]:
try:
    from ultralytics import YOLO
    yseg = YOLO("yolo11n-seg.pt")
    r = yseg.predict(img_bgr, conf=0.25, verbose=False)[0]
    print("Instances:", 0 if r.masks is None else len(r.masks))
    show(r.plot(), titles=["YOLO11n-seg instances"], figsize=(8, 6))
except Exception as e:
    print("YOLO-seg demo skipped:", e)

## 6. Guided example - instance segmentation with Mask R-CNN
Mask R-CNN adds a mask branch to Faster R-CNN. It is accurate and gives per-instance masks and keypoints, but is slower than YOLO-seg.

In [ ]:
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights

mw = MaskRCNN_ResNet50_FPN_Weights.DEFAULT
mask_model = maskrcnn_resnet50_fpn(weights=mw).eval()
mt = mw.transforms()(Image.fromarray(img_rgb)).unsqueeze(0)
with torch.no_grad():
    out = mask_model(mt)[0]

vis = img_rgb.copy()
count = 0
for box, label, score, mask in zip(out["boxes"], out["labels"], out["scores"], out["masks"]):
    if score < 0.6:
        continue
    count += 1
    m = (mask[0].cpu().numpy() > 0.5)
    vis[m] = (0.5 * vis[m] + 0.5 * np.array([0, 255, 0])).astype(np.uint8)
print("Mask R-CNN instances:", count)
show(vis, titles=["Mask R-CNN instance masks"], figsize=(8, 6))

## 7. Guided example - promptable segmentation with SAM (optional)
**SAM** (Segment Anything Model) segments *any* object from a **point, box or text prompt**. It is a foundation model: one model, many tasks. Here we prompt with a point using Ultralytics.

In [ ]:
try:
    from ultralytics import SAM
    sam = SAM("mobile_sam.pt")           # small and fast
    h, w = img_bgr.shape[:2]
    res = sam.predict(img_bgr, points=[[w // 2, h // 2]], labels=[1], verbose=False)[0]
    print("SAM masks:", 0 if res.masks is None else len(res.masks))
    show(res.plot(), titles=["SAM with a single point prompt"], figsize=(8, 6))
except Exception as e:
    print("SAM demo skipped:", e)

###  Interactive exploration - overlay opacity
Adjust how strongly a predicted mask is blended over the image - a practical visualization skill.

In [ ]:
import ipywidgets as widgets
from ipywidgets import interact

if 'colour' in dir():
    def overlay_demo(alpha=0.5):
        ov = cv2.addWeighted(img_rgb, 1 - alpha, colour, alpha, 0)
        show(ov, titles=[f"overlay alpha = {alpha:.2f}"], figsize=(8, 6))
    interact(overlay_demo, alpha=widgets.FloatSlider(min=0.0, max=1.0, step=0.05, value=0.5))
else:
    print("Run the DeepLabV3 cell first.")

## 8. Guided example - segmentation metric IoU / mIoU
$$\mathrm{IoU}_{c} = \frac{|P_c \cap G_c|}{|P_c \cup G_c|}, \qquad \mathrm{mIoU} = \frac{1}{C}\sum_c \mathrm{IoU}_c$$
Unlike pixel accuracy, mIoU is not dominated by large background regions.

In [ ]:
def mask_iou(pred, gt):
    inter = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()
    return inter / union if union else 0.0

rng = np.random.RandomState(0)
for name, p in [("excellent", 0.95), ("good", 0.8), ("fair", 0.55), ("poor", 0.3)]:
    gt = np.zeros((100, 100), bool); gt[20:70, 20:70] = True
    pred = np.zeros((100, 100), bool)
    side = int((70 - 20) * (p ** 0.5))
    pred[25:25 + side, 25:25 + side] = True
    print(f"{name:10s} mask IoU = {mask_iou(pred, gt):.2f}")

## 9. Exercise (complete the code)

Compare **YOLO-seg** and **Mask R-CNN** on the same image:
1. Count the instances each detects at confidence 0.5.
2. Measure inference time for each with `time.perf_counter()`.
3. Fill a table (model, #instances, time) and recommend one for a conveyor-belt inspection camera.

In [ ]:
# TODO: compare YOLO-seg and Mask R-CNN


## 10. Challenge (independent)

Use **SAM** to build a semi-automatic labelling helper:
1. Click (or hard-code) a few points on an object.
2. Generate its mask.
3. Export the mask as a YOLO-seg polygon annotation.

This *model-assisted annotation* workflow is how many modern datasets are built.

In [ ]:
# Your code here


## 11. Check your understanding (Q&A)

<details><summary><b>Q1. What is the difference between semantic and instance segmentation?</b></summary>

Semantic segmentation assigns a class to each pixel but does not separate individual objects of the same class. Instance segmentation produces a distinct mask per object.
</details>

<details><summary><b>Q2. Why do U-Net skip connections matter?</b></summary>

They bring high-resolution spatial detail from the encoder directly to the decoder, enabling precise boundaries. Without them, upsampling loses fine structure.
</details>

<details><summary><b>Q3. Why is mIoU preferred over pixel accuracy?</b></summary>

Pixel accuracy is dominated by large classes (often background). mIoU averages per-class overlap, so small but important classes count equally.
</details>

<details><summary><b>Q4. What makes SAM a 'foundation model'?</b></summary>

It was trained on a massive, diverse mask dataset and generalises to unseen objects and tasks through prompts, rather than being specialised to fixed classes.
</details>

## 12. Further reading & self-exploration
- U-Net paper: Ronneberger et al. (2015).
- DeepLab papers: Chen et al. (2017-2018).
- Mask R-CNN paper: He et al. (2017).
- SAM: https://segment-anything.com/ and https://github.com/facebookresearch/segment-anything
- Ultralytics segment mode: https://docs.ultralytics.com/tasks/segment/
- torchvision segmentation models: https://pytorch.org/vision/stable/models.html
- Wikipedia - Image segmentation: https://en.wikipedia.org/wiki/Image_segmentation

**Try next:** fine-tune U-Net on a small custom mask dataset and compute mIoU per class.

## 13. Key takeaways
- Semantic vs instance vs panoptic segmentation answer different questions.
- **U-Net** = encoder-decoder + skip connections; the standard for dense prediction.
- **YOLO-seg** is fast; **Mask R-CNN** is accurate; **SAM** is promptable and general.
- Evaluate with **IoU / mIoU**.
- Foundation models enable **model-assisted annotation**.